# 🎭 Reconnaissance de Visages Masqués avec ArcFace Loss
## Notebook Complet — Kaggle P100 (corrigé et complet)

### 📚 À Propos
Ce notebook entraîne un modèle de reconnaissance de visages masqués avec **ArcFace Loss**
sur un GPU Kaggle P100 (16 GB VRAM).

### 🔧 Corrections appliquées
- ✅ `train_epoch` et `validate_epoch` **implémentées** (elles étaient absentes)
- ✅ Chemins de sauvegarde corrigés → `/kaggle/working/` (Kaggle, pas Colab)
- ✅ CatMix rendu compatible ArcFace (labels restent entiers)
- ✅ Scheduler réinitialisé à chaque changement de phase
- ✅ `num_workers=2` sur Kaggle (au lieu de 0)
- ✅ Import `resnet50` inutile supprimé
- ✅ `except ImportError` à la place du `except:` nu
- ✅ Cellule bonus rendue autonome
- ✅ **numpy pinned to 1.26.4** — fixes `RuntimeError: Numpy is not available` in DataLoader workers caused by numpy 2.x / numba incompatibility
- ✅ **NumPy sanity check cell** added after installation
- ✅ `persistent_workers` enabled for DataLoaders


## 1️⃣ Installation des Dépendances (Kaggle P100)

In [1]:
import sys, subprocess

print("=" * 70)
print("🔧 PYTORCH INSTALLER FOR KAGGLE P100 (SM 6.0)")
print("=" * 70)

CUDA_INDEX = "https://download.pytorch.org/whl/cu118"

def run_pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--no-cache-dir", "--force-reinstall", *args])

def cuda_smoke_test():
    test_code = r"""
import torch
cc = torch.cuda.get_device_capability(0)
sm = f"sm_{cc[0]}{cc[1]}"
arch_list = torch.cuda.get_arch_list() if hasattr(torch.cuda, "get_arch_list") else []
if arch_list and sm not in arch_list:
    raise RuntimeError(f"No kernel for {sm}. Available: {arch_list}")
bn = torch.nn.BatchNorm2d(8).cuda()
x  = torch.randn(2, 8, 16, 16, device="cuda")
_ = bn(x)
print("cuda_smoke_test=OK")
"""
    subprocess.check_call([sys.executable, "-c", test_code])

# Uninstall existing torch stack
for pkg in ["torch", "torchvision", "torchaudio"]:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", pkg])
    except Exception:
        pass

candidates = [
    ("2.2.0+cu118", "0.17.0+cu118", "2.2.0+cu118"),
    ("2.2.2+cu118", "0.17.2+cu118", "2.2.2+cu118"),
    ("2.3.1+cu118", "0.18.1+cu118", "2.3.1+cu118"),
]

selected = None
for torch_v, tv_v, ta_v in candidates:
    print(f"\n➡️ Trying torch={torch_v}")
    try:
        run_pip_install([f"torch=={torch_v}", f"torchvision=={tv_v}",
                         f"torchaudio=={ta_v}", "--index-url", CUDA_INDEX])
        cuda_smoke_test()
        selected = (torch_v, tv_v, ta_v)
        print(f"   ✅ Build {torch_v} works!")
        break
    except Exception as e:
        print(f"   ❌ Failed: {e}")

if selected is None:
    raise RuntimeError("No compatible torch/cu118 build found for this P100 runtime.")

# Pin numpy to 1.26.4 — numpy>=2.0 breaks numba/DataLoader workers
print("\nPinning numpy to 1.26.4 ...")
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "--no-cache-dir", "numpy==1.26.4"])
print("   ✅ numpy==1.26.4")

# FIX: no empty-string flags — plain clean install for each package
for pkg in ["timm", "scikit-learn", "seaborn", "umap-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--no-cache-dir", pkg])
    print(f"   ✅ {pkg}")

# Re-pin numpy after umap-learn (it may pull a newer version)
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
print("   ✅ numpy re-pinned to 1.26.4")

print("\n⚠️ IMPORTANT: Restart the kernel, then run from Cell 2 onward.")


🔧 PYTORCH INSTALLER FOR KAGGLE P100 (SM 6.0)
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128

➡️ Trying torch=2.2.0+cu118
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.6/811.6 MB 228.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 243.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 243.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 261.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 300.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
nilearn 0.13.1 requires requests>=2.30.0, but you have requests 2.28.1 which is incompatible.
id 1.6.1 requires urllib3<3,>=2, but you have urllib3 1.26.13 which is incompatible.
datasets 4.8.3 requires requests>=2.32.2, but you have requests 2.28.1 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
ydata-profiling 4.18.1 requires requests<3,>=2.32.0, but you have requests 2.28.1 which is incompatible.
jupyterlab-server 2.28.0 requires requests>=2.31, but you have requests 2.28.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you ha


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 2, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 1471, in <module>
    from .functional import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 9, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 1, in <module>
    from .modules import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/__init__.py", line 35, in <modu

cuda_smoke_test=OK
   ✅ Build 2.2.0+cu118 works!

Pinning numpy to 1.26.4 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 234.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
nilearn 0.13.1 requires requests>=2.30.0, but you have requests 2.28.1 which is incompatible.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
datasets 4.8.3 requires requests>=2.32.2, but you have requests 2.28.1 which is incompatible.
ydata-profiling 4.18.1 requires requests<3,>=2.32.0, but you have requests 2.28.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, bu

   ✅ numpy==1.26.4
   ✅ timm
   ✅ scikit-learn
   ✅ seaborn
   ✅ umap-learn
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 326.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ✅ numpy re-pinned to 1.26.4

⚠️ IMPORTANT: Restart the kernel, then run from Cell 2 onward.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
nilearn 0.13.1 requires requests>=2.30.0, but you have requests 2.28.1 which is incompatible.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
datasets 4.8.3 requires requests>=2.32.2, but you have requests 2.28.1 which is incompatible.
ydata-profiling 4.18.1 requires requests<3,>=2.32.0, but you have requests 2.28.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, bu

In [2]:
# ── NumPy version sanity check (run AFTER kernel restart) ────────────────────
# If this cell raises an error, go back and re-run Cell 1, then restart again.
import numpy as np
import torch

np_ver = tuple(int(x) for x in np.__version__.split('.')[:2])
assert np_ver < (2, 0), (
    f"numpy {np.__version__} is too new — DataLoader workers will crash.\n"
    f"Run Cell 1 again, then restart the kernel."
)
print(f"✅ numpy {np.__version__} — compatible (< 2.0)")
print(f"✅ torch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print("✅ Environment looks good — proceed with Cell 3 onwards.")


AssertionError: numpy 2.0.2 is too new — DataLoader workers will crash.
Run Cell 1 again, then restart the kernel.

## 2️⃣ Import des Librairies

In [ ]:
import os, gc, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
from torchvision import transforms
import timm
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.manifold import TSNE

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️ UMAP non disponible, t-SNE sera utilisé à la place")

from tqdm import tqdm
from pathlib import Path
from datetime import datetime
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 70)
print("🖥️  GPU INFORMATION")
print("=" * 70)
print(f"✅ CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU            : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"🔧 CUDA version   : {torch.version.cuda}")
else:
    print("❌ No GPU — CPU will be used (VERY SLOW)")
print("=" * 70)
print("✅ All imports loaded!")


## 3️⃣ Configuration

In [ ]:
import os, torch, numpy as np

# ── Device ──────────────────────────────────────────────────────────────────
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_CUDA_DSA']       = '1'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# ── Config ───────────────────────────────────────────────────────────────────
class Config:
    # Dataset
    USE_SUBSET         = False   # True = 20% of data for quick testing
    SUBSET_PERCENTAGE  = 0.2

    # Image
    image_size         = 224

    # Training
    batch_size         = 64     # Optimised for P100 16 GB
    num_epochs         = 40
    seed               = 42

    # Model
    embedding_dim      = 128
    model_name         = "efficientnet_b0"

    # ArcFace
    arcface_margin     = 0.3    # Reduced for stability
    arcface_scale      = 64

    # Optimiser
    weight_decay       = 1e-4

    # Early stopping
    early_stopping_patience  = 5
    early_stopping_min_delta = 0.001

    # CatMix  (disabled — incompatible with hard-label ArcFace)
    use_catmix   = False
    catmix_alpha = 1.0
    catmix_prob  = 0.5

    # Kaggle output dir  ← FIXED (was /content/drive/MyDrive/)
    output_dir = '/kaggle/working'

# Seed
torch.manual_seed(Config.seed)
np.random.seed(Config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.seed)

print("\n⚙️  CONFIG — KAGGLE P100")
print("=" * 70)
print(f"  Image size   : {Config.image_size}x{Config.image_size}")
print(f"  Batch size   : {Config.batch_size}")
print(f"  Epochs       : {Config.num_epochs}")
print(f"  Embedding dim: {Config.embedding_dim}")
print(f"  ArcFace      : margin={Config.arcface_margin}, scale={Config.arcface_scale}")
print(f"  Subset mode  : {Config.USE_SUBSET}")
print(f"  Output dir   : {Config.output_dir}")
print("=" * 70)


## 4️⃣ Chargement du Dataset Kaggle

In [ ]:
# ── Kaggle dataset path ──────────────────────────────────────────────────────
dataset_path = '/kaggle/input/datasets/chaimaeelmounjali/facesrecognition/Unified_Dataset'

print("📊 Kaggle Dataset Setup")
print("=" * 70)
print(f"📂 Path: {dataset_path}")

if os.path.exists(dataset_path):
    classes = sorted([d for d in os.listdir(dataset_path)
                      if os.path.isdir(os.path.join(dataset_path, d))])
    total_images = 0
    for cls in classes:
        cls_path = os.path.join(dataset_path, cls)
        for sub in ['masked', 'non_masked']:
            sub_path = os.path.join(cls_path, sub)
            if os.path.isdir(sub_path):
                total_images += len([f for f in os.listdir(sub_path)
                                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"   Classes  : {len(classes)}")
    print(f"   Images   : {total_images}")
    if len(classes) > 0:
        print(f"   Avg/class: {total_images // len(classes):.1f}")
    Config.num_classes = len(classes)
    print(f"\n✅ Config.num_classes = {Config.num_classes}")
else:
    # FIX: always define num_classes so downstream cells don't crash with AttributeError
    Config.num_classes = 10   # placeholder — update once dataset is added
    print(f"\n❌ Dataset not found at {dataset_path}")
    print("   Please add the dataset to this Kaggle notebook.")
    print(f"   Config.num_classes set to placeholder={Config.num_classes} — update after adding dataset.")
print("=" * 70)


## 5️⃣ ArcFace Loss

In [ ]:
class ArcFaceLoss(nn.Module):
    """
    ArcFace: Additive Angular Margin Loss
    https://arxiv.org/abs/1801.07698
    """
    def __init__(self, num_classes, embedding_dim=128, margin=0.5, scale=64.0):
        super().__init__()
        self.num_classes   = num_classes
        self.embedding_dim = embedding_dim
        self.margin        = margin
        self.scale         = scale

        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = np.cos(margin)
        self.sin_m = np.sin(margin)
        self.th    = np.cos(np.pi - margin)
        self.mm    = np.sin(np.pi - margin) * margin   # FIX: was sin_m * margin

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, embeddings, labels):
        # Normalise weights
        w = F.normalize(self.weight, p=2, dim=1)

        # cos(theta)
        logits    = F.linear(embeddings, w)            # [B, C]
        batch_idx = torch.arange(embeddings.size(0), device=embeddings.device)

        cos_theta   = logits[batch_idx, labels].unsqueeze(1)          # [B,1]
        sin_theta   = torch.sqrt((1.0 - cos_theta.pow(2)).clamp(0, 1))
        cos_theta_m = cos_theta * self.cos_m - sin_theta * self.sin_m  # cos(θ+m)

        # Safe margin
        cos_theta_m = torch.where(cos_theta > self.th,
                                  cos_theta_m,
                                  cos_theta - self.mm)

        logits[batch_idx, labels] = cos_theta_m.squeeze(1)
        logits = logits * self.scale

        return self.criterion(logits, labels)

print("✅ ArcFaceLoss defined.")


## 6️⃣ Architecture du Modèle (EfficientNet-B0 + Embedding)

In [ ]:
class FaceRecognitionModel(nn.Module):
    """EfficientNet-B0 backbone + L2-normalised embedding head."""

    def __init__(self, embedding_dim=128):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True)
        backbone_out  = self.backbone.num_features          # 1280
        self.backbone.reset_classifier(0)                   # Remove original head
        self.embedding_layer = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features   = self.backbone.forward_features(x)     # [B, 1280, H, W]
        if features.dim() > 2:
            features = features.mean(dim=[2, 3])           # Global avg pool
        embeddings = self.embedding_layer(features)         # [B, emb_dim]
        embeddings = F.normalize(embeddings, p=2, dim=1)   # L2 norm
        return embeddings

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_n_blocks(self, n=2):
        if hasattr(self.backbone, 'blocks'):
            total = len(self.backbone.blocks)
            for i, block in enumerate(self.backbone.blocks):
                for p in block.parameters():
                    p.requires_grad = (i >= total - n)


model = FaceRecognitionModel(embedding_dim=Config.embedding_dim).to(device)

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}")
print("✅ Model ready.")


## 7️⃣ Transformations & Dataset

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((Config.image_size + 32, Config.image_size + 32)),
    transforms.RandomCrop(Config.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_transforms = transforms.Compose([
    transforms.Resize((Config.image_size + 32, Config.image_size + 32)),
    transforms.CenterCrop(Config.image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


class FaceDataset(Dataset):
    """Loads images from root/class/masked/*.jpg  and  root/class/non_masked/*.jpg"""

    def __init__(self, dataset_path, transform=None):
        self.dataset_path = Path(dataset_path)
        self.transform    = transform

        self.classes      = sorted([d for d in os.listdir(dataset_path)
                                    if os.path.isdir(os.path.join(dataset_path, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.images, self.labels = [], []
        for cls_idx, cls_name in enumerate(self.classes):
            cls_path = self.dataset_path / cls_name
            for sub in ['masked', 'non_masked']:
                sub_path = cls_path / sub
                if sub_path.is_dir():
                    for f in sub_path.iterdir():
                        if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                            self.images.append(str(f))
                            self.labels.append(cls_idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        try:
            img   = Image.open(self.images[idx]).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            print(f"⚠️ Error loading {self.images[idx]}: {e}")
            return self.__getitem__((idx + 1) % len(self))


print("✅ Transforms & FaceDataset defined.")


## 8️⃣ Création des DataLoaders (80/20 split)

In [ ]:
print("=" * 70)
print("📂 DATASET PREPARATION")
print("=" * 70)

# ── Load full dataset (no transforms yet) ────────────────────────────────────
full_dataset = FaceDataset(dataset_path, transform=None)
print(f"Total images : {len(full_dataset)}")
print(f"Classes      : {len(full_dataset.classes)}")

# ── Optional subset ──────────────────────────────────────────────────────────
if Config.USE_SUBSET:
    n_sub    = max(1, int(len(full_dataset.classes) * Config.SUBSET_PERCENTAGE))
    sub_cls  = set(full_dataset.classes[:n_sub])
    sub_imgs, sub_lbls = [], []
    for i, (img, lbl) in enumerate(zip(full_dataset.images, full_dataset.labels)):
        if full_dataset.classes[lbl] in sub_cls:
            sub_imgs.append(img)
            sub_lbls.append(lbl)
    # Re-map labels to 0..n_sub-1
    old_idxs = sorted({full_dataset.class_to_idx[c] for c in sub_cls})
    remap    = {old: new for new, old in enumerate(old_idxs)}
    sub_lbls = [remap[l] for l in sub_lbls]

    class _SubFaceDataset(Dataset):
        def __init__(self, imgs, lbls, transform=None):
            self.images, self.labels, self.transform = imgs, lbls, transform
        def __len__(self):
            return len(self.images)
        def __getitem__(self, idx):
            try:
                img = Image.open(self.images[idx]).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                return img, self.labels[idx]
            except Exception:
                return self.__getitem__((idx + 1) % len(self))

    base_ds          = _SubFaceDataset(sub_imgs, sub_lbls)
    Config.num_classes = n_sub
    print(f"SUBSET: {n_sub} classes, {len(sub_imgs)} images")
else:
    base_ds = full_dataset
    Config.num_classes = len(full_dataset.classes)
    print(f"FULL DATASET: {Config.num_classes} classes, {len(base_ds)} images")

# ── Train / Val split ─────────────────────────────────────────────────────────
gen          = torch.Generator().manual_seed(Config.seed)
n_train      = int(0.8 * len(base_ds))
n_val        = len(base_ds) - n_train
train_idx, val_idx = torch.utils.data.random_split(range(len(base_ds)),
                                                    [n_train, n_val], generator=gen)
train_idx, val_idx = list(train_idx), list(val_idx)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")


class _IndexedDataset(Dataset):
    """Wraps a base dataset and applies a transform, indexed by a list."""
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, lbl = self.base[self.indices[i]]
        if self.transform and not isinstance(img, torch.Tensor):
            img = self.transform(img)
        elif self.transform and isinstance(img, torch.Tensor):
            pass   # already a tensor
        return img, lbl


train_dataset = _IndexedDataset(base_ds, train_idx, train_transforms)
val_dataset   = _IndexedDataset(base_ds, val_idx,   val_transforms)

# ── DataLoaders  (num_workers=2 on Kaggle) ─────────────────────────────────
# num_workers: use 2 if numpy>=2 conflict is resolved; fall back to 0 otherwise.
# After the numpy==1.26.4 pin in Cell 1 and a kernel restart, num_workers=2 is safe.
# If you still see "Numpy is not available" errors, set NUM_WORKERS = 0.
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=Config.batch_size,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_dataset,   batch_size=Config.batch_size,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print("✅ DataLoaders ready.")


## 9️⃣ Initialisation de l'Entraînement

In [ ]:
# ── ArcFace criterion ────────────────────────────────────────────────────────
criterion = ArcFaceLoss(
    num_classes   = Config.num_classes,
    embedding_dim = Config.embedding_dim,
    margin        = Config.arcface_margin,
    scale         = Config.arcface_scale,
).to(device)

# ── Optimiser ────────────────────────────────────────────────────────────────
def make_optimizer(lr):
    """Create a fresh AdamW with a given learning rate."""
    return AdamW(
        list(model.parameters()) + list(criterion.parameters()),
        lr=lr,
        weight_decay=Config.weight_decay,
    )

optimizer = make_optimizer(lr=0.001)

# ── Scheduler — will be re-created at each phase change ──────────────────────
def make_scheduler(opt, t_max, eta_min=1e-5):
    return CosineAnnealingLR(opt, T_max=t_max, eta_min=eta_min)

scheduler = make_scheduler(optimizer, t_max=3)   # Phase 1: 3 epochs

# ── History ───────────────────────────────────────────────────────────────────
history = {'train_loss': [], 'train_accuracy': [],
           'val_loss': [],   'val_accuracy': [],
           'learning_rate': []}

best_val_accuracy    = 0.0
early_stopping_counter = 0
best_model_state     = None
optimal_threshold    = 0.5

print("✅ Criterion, optimiser and scheduler created.")
print(f"   ArcFace margin={Config.arcface_margin}, scale={Config.arcface_scale}")
print(f"   Num classes={Config.num_classes}")


## 🔟 Fonctions `train_epoch` et `validate_epoch`
*(Ces fonctions étaient absentes dans la version originale — bug critique corrigé)*

In [ ]:
def train_epoch(epoch):
    """One full training pass. Returns (avg_loss, accuracy_%)."""
    model.train()
    criterion.train()

    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Train Ep{epoch+1}", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        embeddings = model(images)                    # [B, emb_dim]
        loss       = criterion(embeddings, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(criterion.parameters()), max_norm=1.0)
        optimizer.step()

        # Accuracy (cosine nearest-class)
        with torch.no_grad():
            w       = F.normalize(criterion.weight, p=2, dim=1)
            logits  = F.linear(embeddings, w) * Config.arcface_scale
            preds   = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

        total_loss += loss.item() * labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


@torch.no_grad()
def validate_epoch(epoch):
    """One full validation pass.
    Returns (avg_loss, accuracy_%, embeddings_cpu, labels_cpu)."""
    model.eval()
    criterion.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_emb, all_lbl = [], []

    pbar = tqdm(val_loader, desc=f"Val   Ep{epoch+1}", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        embeddings = model(images)
        loss       = criterion(embeddings, labels)

        w      = F.normalize(criterion.weight, p=2, dim=1)
        logits = F.linear(embeddings, w) * Config.arcface_scale
        preds  = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        total_loss += loss.item() * labels.size(0)

        all_emb.append(embeddings.cpu())
        all_lbl.append(labels.cpu())

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy, torch.cat(all_emb), torch.cat(all_lbl)


print("✅ train_epoch and validate_epoch defined.")


## 1️⃣1️⃣ Validation GPU avant Entraînement

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU (CUDA) is required. Enable it in Kaggle settings.")

cc         = torch.cuda.get_device_capability(0)
req_sm     = f"sm_{cc[0]}{cc[1]}"
arch_list  = torch.cuda.get_arch_list() if hasattr(torch.cuda, "get_arch_list") else []
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"CC   : {cc[0]}.{cc[1]}  ({req_sm})")
if arch_list and req_sm not in arch_list:
    raise RuntimeError(f"No kernels for {req_sm}. Wheel supports {arch_list}.")

# Quick forward-pass smoke test
with torch.no_grad():
    _x = torch.randn(2, 3, Config.image_size, Config.image_size, device='cuda')
    _l = torch.tensor([0, 1], device='cuda')
    _e = model(_x)
    _s = criterion(_e, _l)
print(f"✅ GPU smoke test passed. Loss={_s.item():.4f}")
del _x, _l, _e, _s
torch.cuda.empty_cache()


## 1️⃣2️⃣ Entraînement en 3 Phases

In [ ]:
print("=" * 70)
print("🚀 TRAINING — 3 PHASES")
print("=" * 70)

current_phase = None

for epoch in range(Config.num_epochs):

    # ── Phase transitions (scheduler re-created each time) ──────────────────
    if epoch < 3:
        phase = 1
        if current_phase != 1:
            current_phase = 1
            model.freeze_backbone()
            for p in model.embedding_layer.parameters():
                p.requires_grad = True
            optimizer = make_optimizer(lr=1e-3)
            scheduler = make_scheduler(optimizer, t_max=3)
            print("\n📍 PHASE 1 (Ep 1-3): Freeze backbone — LR=1e-3")

    elif epoch < 8:
        phase = 2
        if current_phase != 2:
            current_phase = 2
            model.unfreeze_last_n_blocks(n=2)
            optimizer = make_optimizer(lr=1e-4)
            scheduler = make_scheduler(optimizer, t_max=5)
            print("\n📍 PHASE 2 (Ep 4-8): Unfreeze last 2 blocks — LR=1e-4")

    else:
        phase = 3
        if current_phase != 3:
            current_phase = 3
            model.unfreeze_backbone()
            optimizer = make_optimizer(lr=5e-5)
            scheduler = make_scheduler(optimizer, t_max=Config.num_epochs - 8, eta_min=1e-6)
            print("\n📍 PHASE 3 (Ep 9+): Full model — LR=5e-5")

    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{Config.num_epochs}  |  Phase {phase}")

    train_loss, train_acc              = train_epoch(epoch)
    val_loss, val_acc, val_emb, val_lbl = validate_epoch(epoch)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])

    print(f"  Train  loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val    loss={val_loss:.4f}  acc={val_acc:.2f}%")
    print(f"  LR = {optimizer.param_groups[0]['lr']:.2e}")

    # ── Early stopping ───────────────────────────────────────────────────────
    if val_acc >= best_val_accuracy + Config.early_stopping_min_delta:
        best_val_accuracy = val_acc
        early_stopping_counter = 0
        best_model_state = {
            'model'       : {k: v.cpu() for k, v in model.state_dict().items()},
            'criterion'   : {k: v.cpu() for k, v in criterion.state_dict().items()},
            'epoch'       : epoch,
            'val_accuracy': val_acc,
            'val_emb'     : val_emb,
            'val_lbl'     : val_lbl,
        }
        print(f"  ✅ New best model! acc={val_acc:.2f}%")
    else:
        early_stopping_counter += 1
        print(f"  ⏱️ No improvement ({early_stopping_counter}/{Config.early_stopping_patience})")
        if early_stopping_counter >= Config.early_stopping_patience:
            print(f"\n⏹️ Early stopping at epoch {epoch+1}")
            break

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE")
print(f"   Best val accuracy : {best_val_accuracy:.2f}%")
print(f"   Epochs completed  : {epoch+1}/{Config.num_epochs}")
print("=" * 70)


## 1️⃣3️⃣ Chargement du Meilleur Modèle & Évaluation

In [ ]:
if best_model_state is None:
    raise RuntimeError("No model was saved — check training loop.")

model.load_state_dict(best_model_state['model'])
criterion.load_state_dict(best_model_state['criterion'])

best_epoch    = best_model_state['epoch']
best_accuracy = best_model_state['val_accuracy']
emb_np        = best_model_state['val_emb'].numpy()
lbl_np        = best_model_state['val_lbl'].numpy()

print(f"✅ Best model loaded  (epoch {best_epoch+1}, val acc={best_accuracy:.2f}%)")

# ── Accuracy from logits ──────────────────────────────────────────────────────
w_np        = F.normalize(criterion.weight, p=2, dim=1).detach().cpu().numpy()
logits_np   = emb_np @ w_np.T * Config.arcface_scale
pred_lbl    = logits_np.argmax(axis=1)

# ── Pairwise ROC ─────────────────────────────────────────────────────────────
n   = len(emb_np)
max_pairs = min(20_000, n * (n - 1) // 2)
rng = np.random.default_rng(Config.seed)

pair_scores, pair_labels = [], []
for i in rng.choice(n, size=min(n, 300), replace=False):
    for j in range(i + 1, n):
        if len(pair_scores) >= max_pairs:
            break
        sim = float(np.dot(emb_np[i], emb_np[j]) /
                    (np.linalg.norm(emb_np[i]) * np.linalg.norm(emb_np[j]) + 1e-8))
        pair_scores.append(sim)
        pair_labels.append(1 if lbl_np[i] == lbl_np[j] else 0)
    if len(pair_scores) >= max_pairs:
        break

pair_scores = np.array(pair_scores)
pair_labels = np.array(pair_labels)

fpr, tpr, thresholds = roc_curve(pair_labels, pair_scores)
auc_score = auc(fpr, tpr)

far_arr  = fpr
frr_arr  = 1 - tpr
opt_idx  = np.argmin(far_arr + frr_arr)
optimal_threshold = float(thresholds[opt_idx])
false_accepts     = float(far_arr[opt_idx])
false_rejects     = float(frr_arr[opt_idx])

print("\n📊 EVALUATION METRICS")
print("=" * 70)
print(f"  Accuracy        : {best_accuracy:.2f}%")
print(f"  AUC             : {auc_score:.4f}")
print(f"  Optimal threshold: {optimal_threshold:.4f}")
print(f"  FAR @ threshold : {false_accepts*100:.2f}%")
print(f"  FRR @ threshold : {false_rejects*100:.2f}%")
print("=" * 70)


## 1️⃣4️⃣ Visualisations

In [ ]:
out = Config.output_dir
os.makedirs(out, exist_ok=True)

# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves — 3 Phases', fontsize=16, fontweight='bold')
epochs_done = range(1, len(history['train_loss']) + 1)

for ax, (tr, va, title, ylabel) in zip(axes, [
    (history['train_loss'],     history['val_loss'],     'Loss',          'Loss'),
    (history['train_accuracy'], history['val_accuracy'], 'Accuracy',      'Accuracy (%)'),
    (history['learning_rate'],  None,                    'Learning Rate', 'LR'),
]):
    ax.plot(epochs_done, tr, label='Train', linewidth=2)
    if va is not None:
        ax.plot(epochs_done, va, label='Val', linewidth=2)
    for xv, col, lbl in [(3,'red','Ph1→2'),(8,'orange','Ph2→3')]:
        if xv < len(history['train_loss']):
            ax.axvline(x=xv, color=col, linestyle='--', alpha=0.6, label=lbl)
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
    if ylabel == 'LR':
        ax.set_yscale('log')

plt.tight_layout()
plt.savefig(f'{out}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ training_curves.png saved")

# ── ROC Curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, linewidth=2.5, label=f'ROC (AUC={auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.scatter(false_accepts, 1 - false_rejects, color='red', s=100, zorder=5,
           label=f'Optimal threshold={optimal_threshold:.3f}')
ax.set_xlabel('False Accept Rate (FAR)')
ax.set_ylabel('True Accept Rate (TAR)')
ax.set_title('ROC Curve — Face Recognition', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{out}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ roc_curve.png saved")

# ── Confusion matrix (≤20 classes) ───────────────────────────────────────────
if Config.num_classes <= 20:
    cm = confusion_matrix(lbl_np, pred_lbl)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{out}/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ confusion_matrix.png saved")

# ── t-SNE ─────────────────────────────────────────────────────────────────────
print("\n🔄 Computing t-SNE (may take a few minutes)...")
max_tsne = min(2000, len(emb_np))
idx_tsne = np.random.choice(len(emb_np), max_tsne, replace=False)
tsne     = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
emb_2d   = tsne.fit_transform(emb_np[idx_tsne])

fig, ax = plt.subplots(figsize=(12, 9))
sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                c=lbl_np[idx_tsne], cmap='tab20', s=20, alpha=0.7)
ax.set_title('t-SNE of Validation Embeddings', fontsize=14, fontweight='bold')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.colorbar(sc, ax=ax, label='Class ID')
plt.tight_layout()
plt.savefig(f'{out}/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ tsne_embeddings.png saved")


## 1️⃣5️⃣ Sauvegarde du Modèle

In [ ]:
out = Config.output_dir
os.makedirs(out, exist_ok=True)

save_dict = {
    'model_state_dict'    : best_model_state['model'],
    'criterion_state_dict': best_model_state['criterion'],
    'config': {
        'image_size'    : Config.image_size,
        'embedding_dim' : Config.embedding_dim,
        'num_classes'   : Config.num_classes,
        'arcface_margin': Config.arcface_margin,
        'arcface_scale' : Config.arcface_scale,
        'batch_size'    : Config.batch_size,
        'seed'          : Config.seed,
    },
    'metrics': {
        'best_epoch'        : best_epoch,
        'val_accuracy'      : float(best_accuracy),
        'auc_score'         : float(auc_score),
        'optimal_threshold' : float(optimal_threshold),
        'far'               : float(false_accepts),
        'frr'               : float(false_rejects),
    },
    'history': {k: [float(v) for v in vals] for k, vals in history.items()},
}

# Full checkpoint
ckpt_path = f'{out}/best_arcface_model.pth'
torch.save(save_dict, ckpt_path)
print(f"✅ Full checkpoint  : {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

# Weights only
w_path = f'{out}/best_arcface_weights_only.pth'
torch.save(best_model_state['model'], w_path)
print(f"✅ Weights only     : {w_path}  ({os.path.getsize(w_path)/1e6:.1f} MB)")

# JSON
for name, data in [('model_config', save_dict['config']),
                   ('model_metrics', save_dict['metrics'])]:
    p = f'{out}/{name}.json'
    with open(p, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"✅ {name}.json saved")


## 1️⃣6️⃣ Résumé Final

In [ ]:
print("\n" + "🎉" * 35)
print("\n" + "=" * 80)
print(" " * 25 + "📋 FINAL SUMMARY")
print("=" * 80)

print(f"\n📊 DATASET")
print(f"   Path       : {dataset_path}")
print(f"   Classes    : {Config.num_classes}")
print(f"   Train imgs : {len(train_idx)}")
print(f"   Val imgs   : {len(val_idx)}")

print(f"\n🏗️  ARCHITECTURE")
print(f"   Backbone   : EfficientNet-B0 (ImageNet pre-trained)")
print(f"   Emb dim    : {Config.embedding_dim}")
print(f"   Loss       : ArcFace (margin={Config.arcface_margin}, scale={Config.arcface_scale})")
print(f"   Norm       : L2 normalisation")

print(f"\n⚙️  TRAINING")
print(f"   Batch size : {Config.batch_size}")
print(f"   Epochs     : {len(history['train_loss'])}/{Config.num_epochs}")
print(f"   Seed       : {Config.seed}")
print(f"   Phases     : 3 (freeze → unfreeze last 2 → full, each with own scheduler)")

print(f"\n📈 RESULTS")
print(f"   Best epoch : {best_epoch + 1}")
print(f"   Val acc    : {best_accuracy:.2f}%")
print(f"   AUC        : {auc_score:.4f}")
print(f"   Threshold  : {optimal_threshold:.4f}")
print(f"   FAR        : {false_accepts*100:.2f}%")
print(f"   FRR        : {false_rejects*100:.2f}%")

print(f"\n💾 FILES  →  {Config.output_dir}/")
for fname in ['best_arcface_model.pth', 'best_arcface_weights_only.pth',
              'model_config.json', 'model_metrics.json',
              'training_curves.png', 'roc_curve.png',
              'tsne_embeddings.png']:
    p = f"{Config.output_dir}/{fname}"
    status = "✅" if os.path.exists(p) else "❌"
    print(f"   {status} {fname}")

print("\n" + "=" * 80)
print(" " * 20 + "🎊 TRAINING COMPLETE — MODEL READY! 🎊")
print("=" * 80)
print("\n" + "🎉" * 35)


## 🎁 BONUS — Charger et Utiliser le Modèle Sauvegardé

Ce bloc est **autonome** : copiez-le dans n'importe quel autre notebook.


In [ ]:
# ── Self-contained inference example ─────────────────────────────────────────
# Requires: torch, timm, torchvision, PIL  (all available on Kaggle/Colab)

import os, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from torchvision import transforms
from PIL import Image
import timm

# ── Re-define the classes needed for loading ──────────────────────────────────

class _ArcFaceLoss(nn.Module):
    def __init__(self, num_classes, embedding_dim, margin, scale):
        super().__init__()
        self.scale  = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.criterion = nn.CrossEntropyLoss()
    def forward(self, emb, labels):
        w = F.normalize(self.weight, p=2, dim=1)
        return self.criterion(F.linear(emb, w) * self.scale, labels)

class _FaceModel(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=False)
        # FIX: read num_features BEFORE reset_classifier (consistent with Cell 13)
        backbone_out = self.backbone.num_features
        self.backbone.reset_classifier(0)
        self.embedding_layer = nn.Sequential(
            nn.Linear(backbone_out, emb_dim),
            nn.BatchNorm1d(emb_dim)
        )
    def forward(self, x):
        f = self.backbone.forward_features(x)
        if f.dim() > 2:
            f = f.mean(dim=[2, 3])
        return F.normalize(self.embedding_layer(f), p=2, dim=1)


def load_model(checkpoint_path, device='cpu'):
    # FIX: weights_only=False silences FutureWarning in torch 2.x
    ckpt  = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg   = ckpt['config']
    mdl   = _FaceModel(cfg['embedding_dim']).to(device)
    crit  = _ArcFaceLoss(cfg['num_classes'], cfg['embedding_dim'],
                          cfg['arcface_margin'], cfg['arcface_scale']).to(device)
    mdl.load_state_dict(ckpt['model_state_dict'])
    crit.load_state_dict(ckpt['criterion_state_dict'])
    mdl.eval(); crit.eval()
    thresh = ckpt['metrics']['optimal_threshold']
    return mdl, crit, cfg, thresh


def predict(model, criterion, cfg, image_path, device='cpu'):
    _tfm = transforms.Compose([
        transforms.Resize((cfg['image_size'] + 32,) * 2),
        transforms.CenterCrop(cfg['image_size']),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = _tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb    = model(tensor)
        w      = F.normalize(criterion.weight, p=2, dim=1)
        logits = F.linear(emb, w) * cfg['arcface_scale']
        conf, cls = torch.max(logits, 1)
    return int(cls.item()), float(conf.item())


# ── Demo (update path before running) ────────────────────────────────────────
CKPT = '/kaggle/working/best_arcface_model.pth'
IMG  = 'path/to/your/test_image.jpg'   # ← update this

if os.path.exists(CKPT):
    _model, _crit, _cfg, _thresh = load_model(CKPT)
    print(f"✅ Model loaded from {CKPT}")
    print(f"   Classes={_cfg['num_classes']}, emb_dim={_cfg['embedding_dim']}")
    print(f"   Optimal threshold={_thresh:.4f}")

    if os.path.exists(IMG):
        cls_id, confidence = predict(_model, _crit, _cfg, IMG)
        print(f"\n🔍 Prediction: class {cls_id}  |  confidence={confidence:.4f}")
    else:
        print(f"\n(Set IMG to a real image path to run inference)")
else:
    print(f"Run the training cells first to generate {CKPT}")
